# ETF Backtest Starter Kit

**Level:** Beginner  
**Tags:** Backtest, ETF, Strategy

## Overview

Load ETF prices, build simple strategy, evaluate performance

## References

1. Hull, J.C. (2022). *Options, Futures, and Other Derivatives*. Pearson.
2. Shreve, S.E. (2004). *Stochastic Calculus for Finance II*. Springer.
3. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*. Wiley.

## Learning Objectives

**Estimated Time**: 120-150 minutes

By the end of this notebook, you will be able to:

1. Build a complete ETF backtesting framework from scratch
2. Calculate performance metrics (returns, Sharpe ratio, drawdowns)
3. Implement portfolio rebalancing strategies
4. Visualize backtest results with professional charts


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Additional imports
try:
    import yfinance as yf
    print('yfinance imported successfully')
except ImportError:
    print('yfinance not installed. Install with: pip install yfinance')
    yf = None

from datetime import datetime, timedelta
import time

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
np.random.seed(42)

print('Libraries imported successfully!')

## 1. Introduction to ETF Backtest Starter Kit

Backtesting is the process of testing a trading strategy on historical data to evaluate its performance before implementing it in live markets. ETFs (Exchange-Traded Funds) are popular investment vehicles that track indices, sectors, or strategies.

This notebook introduces:
- Loading historical ETF price data
- Implementing simple trading strategies
- Calculating performance metrics
- Evaluating strategy risk and returns
- Visualizing backtest results

We'll build buy-and-hold and moving average crossover strategies for popular ETFs like SPY (S&P 500) and QQQ (Nasdaq 100).

### Installing Required Libraries

If not already installed, run the following command:

```bash
!pip install yfinance pandas numpy matplotlib seaborn scipy
```

- **yfinance**: For fetching ETF historical data
- **Other libraries**: pandas, numpy, matplotlib, seaborn, scipy (already listed above)

In [ ]:
# Function to load ETF data
def load_etf_data(tickers, start_date='2020-01-01', end_date=None):
    """
    Load historical price data for ETFs.
    
    Parameters:
    - tickers: list of ETF tickers
    - start_date: start date for data
    - end_date: end date for data
    
    Returns:
    - pd.DataFrame with Close prices for each ETF
    """
    if yf is None:
        print('yfinance not available')
        return None
    
    if end_date is None:
        end_date = datetime.today().strftime('%Y-%m-%d')
    
    try:
        data = yf.download(tickers, start=start_date, end=end_date)
        close_prices = data['Close']
        print(f'Loaded data for {len(tickers)} ETFs from {start_date} to {end_date}')
        return close_prices
    except Exception as e:
        print(f'Error loading data: {e}')
        return None

# Test data loading
etf_tickers = ['SPY', 'QQQ', 'VWO', 'BND']
if yf is not None:
    etf_data = load_etf_data(etf_tickers, '2020-01-01')
    print(etf_data.head())
else:
    # Create synthetic data for demonstration
    dates = pd.date_range('2020-01-01', periods=800, freq='D')
    np.random.seed(42)
    etf_data = pd.DataFrame({
        'SPY': 300 + np.cumsum(np.random.normal(0.0005, 0.015, 800)) * 300,
        'QQQ': 280 + np.cumsum(np.random.normal(0.0007, 0.025, 800)) * 280,
        'VWO': 40 + np.cumsum(np.random.normal(0.0003, 0.02, 800)) * 40,
        'BND': 80 + np.cumsum(np.random.normal(0.0001, 0.005, 800)) * 80
    }, index=dates)
    etf_data = etf_data.dropna()
    print('Using synthetic ETF data for demonstration')

## 2. Mathematical Foundation

### Performance Metrics

**Total Return:**
$$R = \frac{P_T - P_0}{P_0} \times 100$$

where $P_T$ is the final price and $P_0$ is the initial price.

**Annualized Return:**
$$R_{annual} = \left( \frac{P_T}{P_0} \right)^{\frac{365}{T}} - 1$$

where T is the number of days.

**Volatility (Standard Deviation):**
$$\sigma = \sqrt{\frac{\sum (r_i - \bar{r})^2}{n-1}}$$
where $r_i$ are daily returns.

**Sharpe Ratio:**
$$SR = \frac{R_{annual} - r_f}{\sigma_{annual}}$$

where $r_f$ is risk-free rate.

**Maximum Drawdown:**
$$MDD = \max_{t \in [0,T]} \left( \frac{P_t - \max_{s \in [0,t]} P_s}{\max_{s \in [0,t]} P_s} \right)$$

### Strategy Signals

**Moving Average Crossover:**
- Buy when short MA crosses above long MA
- Sell when short MA crosses below long MA

$$MA_{short}(t) > MA_{long}(t) \rightarrow Buy$$
$$MA_{short}(t) < MA_{long}(t) \rightarrow Sell$$

## 3. Python Implementation

Let's implement backtesting functions for simple trading strategies.

In [ ]:
# Performance calculation functions
def calculate_returns(prices):
    """Calculate percentage returns"""
    return prices.pct_change().dropna()

def annualized_return(prices):
    """Calculate annualized return"""
    total_return = prices.iloc[-1] / prices.iloc[0] - 1
    days = (prices.index[-1] - prices.index[0]).days
    return (1 + total_return) ** (365 / days) - 1

def volatility(returns):
    """Calculate annualized volatility"""
    return returns.std() * np.sqrt(252)  # 252 trading days

def sharpe_ratio(returns, risk_free_rate=0.02):
    """Calculate Sharpe ratio (annualized)"""
    excess_returns = returns.mean() * 252 - risk_free_rate
    vol = volatility(returns)
    return excess_returns / vol if vol > 0 else 0

def max_drawdown(prices):
    """Calculate maximum drawdown"""
    peak = prices.expanding().max()
    drawdown = (prices - peak) / peak
    return drawdown.min()

# Strategy implementations
def buy_and_hold(prices):
    """Buy and hold strategy"""
    positions = pd.Series(1.0, index=prices.index)  # Always 100% invested
    return positions

def moving_average_crossover(prices, short_window=20, long_window=50):
    """Moving average crossover strategy"""
    short_ma = prices.rolling(window=short_window).mean()
    long_ma = prices.rolling(window=long_window).mean()
    
    signals = pd.Series(0, index=prices.index)
    signals[short_ma > long_ma] = 1  # Buy signal
    signals[short_ma <= long_ma] = 0  # Sell signal
    
    return signals

def backtest_strategy(prices, positions):
    """Backtest a strategy given positions (0 or 1)"""
    returns = calculate_returns(prices)
    strategy_returns = returns * positions.shift(1).fillna(0)  # Shift to avoid lookahead bias
    
    cumulative_returns = (1 + strategy_returns).cumprod()
    cumulative_returns.iloc[0] = 1.0  # Initial value
    
    return strategy_returns, cumulative_returns

print('Backtesting functions defined')

## 4. Visualization and Analysis

Let's run backtests and visualize the results.

In [ ]:
# Run backtests for multiple ETFs
etf_results = {}

for etf in etf_data.columns:
    prices = etf_data[etf].dropna()
    returns = calculate_returns(prices)
    
    # Strategies
    bh_positions = buy_and_hold(prices)
    ma_positions = moving_average_crossover(prices)
    
    # Backtest results
    bh_strategy_returns, bh_cumulative = backtest_strategy(prices, bh_positions)
    ma_strategy_returns, ma_cumulative = backtest_strategy(prices, ma_positions)
    
    # Performance metrics
    bh_metrics = {
        'total_return': (bh_cumulative.iloc[-1] - 1) * 100,
        'annualized_return': annualized_return(prices) * 100,
        'volatility': volatility(returns) * 100,
        'sharpe_ratio': sharpe_ratio(returns),
        'max_drawdown': max_drawdown(prices) * 100
    }
    
    ma_metrics = {
        'total_return': (ma_cumulative.iloc[-1] - 1) * 100,
        'annualized_return': annualized_return(prices[ma_positions.dropna().index]) * 100 if len(ma_positions.dropna()) > 0 else 0,
        'volatility': ma_strategy_returns.std() * np.sqrt(252) * 100,
        'sharpe_ratio': sharpe_ratio(ma_strategy_returns),
        'max_drawdown': max_drawdown(ma_cumulative) * 100
    }
    
    etf_results[etf] = {
        'bh_cumulative': bh_cumulative,
        'ma_cumulative': ma_cumulative,
        'bh_metrics': bh_metrics,
        'ma_metrics': ma_metrics
    }

# Plot results for SPY
if 'SPY' in etf_results:
    spy_results = etf_results['SPY']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Price and strategies
    axes[0,0].plot(etf_data.index, etf_data['SPY'], label='Price', alpha=0.7)
    axes[0,0].plot(spy_results['bh_cumulative'].index, spy_results['bh_cumulative'] * etf_data['SPY'].iloc[0], 
                   label='Buy & Hold Portfolio Value', linewidth=2)
    axes[0,0].set_title('SPY Price and Portfolio Value')
    axes[0,0].set_ylabel('Value ($)')
    axes[0,0].legend()
    
    # Cumulative returns comparison
    axes[0,1].plot(spy_results['bh_cumulative'].index, spy_results['bh_cumulative'], label='Buy & Hold', linewidth=2)
    axes[0,1].plot(spy_results['ma_cumulative'].index, spy_results['ma_cumulative'], label='MA Crossover', linewidth=2, color='orange')
    axes[0,1].set_title('Strategy Comparison - Cumulative Returns')
    axes[0,1].set_ylabel('Cumulative Return')
    axes[0,1].legend()
    
    # Returns distribution
    bh_returns = calculate_returns(etf_data['SPY'])
    ma_returns_clean = spy_results['ma_cumulative'].pct_change().dropna()
    
    if len(bh_returns) > 1:
        sns.histplot(bh_returns, ax=axes[1,0], alpha=0.7, label='Buy & Hold', kde=True)
    if len(ma_returns_clean) > 1:
        sns.histplot(ma_returns_clean, ax=axes[1,0], alpha=0.7, label='MA Crossover', kde=True, color='orange')
    axes[1,0].set_title('Returns Distribution')
    axes[1,0].set_xlabel('Daily Return')
    axes[1,0].legend()
    
    # Performance metrics summary
    metrics_df = pd.DataFrame({
        'Buy & Hold': spy_results['bh_metrics'],
        'MA Crossover': spy_results['ma_metrics']
    })
    
    axes[1,1].axis('off')
    table_data = []
    for metric in ['Total Return (%)', 'Annualized Return (%)', 'Volatility (%)', 'Sharpe Ratio', 'Max Drawdown (%)']:
        row = [metric]
        if metric == 'Total Return (%)':
            row.extend([f"{spy_results['bh_metrics']['total_return']:.2f}", f"{spy_results['ma_metrics']['total_return']:.2f}"])
        elif metric == 'Annualized Return (%)':
            row.extend([f"{spy_results['bh_metrics']['annualized_return']:.2f}", f"{spy_results['ma_metrics']['annualized_return']:.2f}"])
        elif metric == 'Volatility (%)':
            row.extend([f"{spy_results['bh_metrics']['volatility']:.2f}", f"{spy_results['ma_metrics']['volatility']:.2f}"])
        elif metric == 'Sharpe Ratio':
            row.extend([f"{spy_results['bh_metrics']['sharpe_ratio']:.2f}", f"{spy_results['ma_metrics']['sharpe_ratio']:.2f}"])
        elif metric == 'Max Drawdown (%)':
            row.extend([f"{spy_results['bh_metrics']['max_drawdown']:.2f}", f"{spy_results['ma_metrics']['max_drawdown']:.2f}"])
        table_data.append(row)
    
    table = axes[1,1].table(cellText=table_data, colLabels=['Metric', 'Buy & Hold', 'MA Crossover'], 
                           loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)
    axes[1,1].set_title('Performance Metrics Summary')
    
    plt.tight_layout()
    plt.show()
    
    print('\nPerformance Summary for SPY:')
    print(metrics_df.round(4))

## 5. Practical Applications

ETF backtesting is fundamental for:

- **Strategy Development**: Test trading ideas before live trading
- **Portfolio Construction**: Evaluate ETF allocation strategies
- **Risk Management**: Assess strategy volatility and drawdown risks
- **Performance Attribution**: Understand what drives returns
- **Investment Research**: Compare ETFs and strategies objectively

**Best Practices:**
- Use sufficient historical data for reliable results
- Account for transaction costs and slippage in backtests
- Avoid survivorship bias by using all available ETFs
- Validate strategies on out-of-sample data
- Consider market conditions and regime changes
- Use realistic position sizing and risk limits

## Summary

This notebook covered ETF backtesting:

- Loaded historical ETF price data using yfinance
- Implemented buy-and-hold and moving average crossover strategies
- Calculated key performance metrics (returns, volatility, Sharpe ratio, drawdown)
- Created comprehensive visualizations of strategy performance
- Demonstrated practical backtesting workflow

Key takeaways:
- Backtesting helps validate trading strategies before live implementation
- Multiple metrics needed to evaluate strategy performance comprehensively
- Simple strategies like buy-and-hold often outperform complex active strategies
- Risk-adjusted metrics like Sharpe ratio are crucial for strategy comparison
- Proper data handling and position management is essential for realistic backtests

### Next Steps
- Add transaction costs and slippage to make backtests more realistic
- Implement more complex strategies (momentum, mean-reversion)
- Create multi-ETF portfolios and portfolio optimization
- Backtest with different market conditions and time periods
- Add risk management overlays (stop-losses, position limits)

### Additional Resources

- [yfinance Documentation](https://pypi.org/project/yfinance/)
- [Backtrader](https://www.backtrader.com/) - Python backtesting framework
- [PyPortfolioOpt](https://github.com/robertmartin8/PyPortfolioOpt) - Portfolio optimization
- [QuantConnect](https://www.quantconnect.com/) - Cloud-based backtesting platform
- [QuantEdX Courses](https://www.quantedx.com/courses)
- [Community Forum](https://www.quantedx.com/community)
- [GitHub Repository](https://github.com/quantedx)